# RAG: черновой ноутбук

Пробные вызовы пайплайнов из `rag/` и сводка метрик по CSV в `evaluation_results/`.

Индексация: `scripts/index_to_qdrant.py`. Полная оценка: `evaluate_rag.py`. Нужен `.env` в корне репозитория.

## Настройка

In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv

EVAL_DIR = Path.cwd()
if not (EVAL_DIR / "evaluate_rag.py").exists():
    EVAL_DIR = Path.cwd() / "evaluation"
PROJECT_ROOT = EVAL_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT.resolve()))

from evaluate_rag import Settings
from rag import build_llm, build_retrieval_embeddings
from rag.config import build_qdrant_client
from rag.retriever import QdrantRetriever

TOP_K_DOCS = 10
CANDIDATE_K = 20

load_dotenv(PROJECT_ROOT / ".env")
settings = Settings()

client = build_qdrant_client(url=settings.qdrant_url, api_key=settings.qdrant_api_key)
embeddings = build_retrieval_embeddings()
llm = build_llm()

/Users/trvexe/Desktop/book-rag/rag/llm.py:10: UserWarning: yandex-cloud-ml-sdk package is deprecated; use yandex-ai-studio-sdk package instead and refer to https://github.com/yandex-cloud/yandex-ai-studio-sdk/blob/master/compat/yandex-cloud-ml-sdk/MIGRATION.md
  from yandex_cloud_ml_sdk import YCloudML
/Users/trvexe/Desktop/book-rag/rag/retriever.py:13: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(
/Users/trvexe/Desktop/book-rag/venv/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:288: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_w

## Пробные запросы RAG

Коллекция Qdrant должна быть уже проиндексирована (`scripts/index_to_qdrant.py`).

In [12]:
COLLECTION_NAME = "prestuplenie_user_bge_m3"
BOOK_TITLE = "Преступление и наказание"
TEST_QUESTION = "Как Соня реагирует на признание Раскольникова в убийстве?"

retriever = QdrantRetriever(client, COLLECTION_NAME, embeddings)


def print_rag_preview(title: str, result: dict, *, max_chars: int = 900) -> None:
    print(f"\n=== {title} ===")
    print("\nQUESTION:", TEST_QUESTION)
    print("\nANSWER:")
    print((result.get("answer") or "")[:max_chars])
    ctx = result.get("context") or []
    print(f"\nCONTEXT: {len(ctx)} chunk(s), first preview:")
    if ctx:
        print((ctx[0] or "")[:max_chars])

### Vanilla RAG

In [13]:
from rag import VanillaRAG

vanilla_rag = VanillaRAG(llm, retriever, TOP_K_DOCS, BOOK_TITLE)
vanilla_result = vanilla_rag.run_rag_pipeline(question=TEST_QUESTION)
print_rag_preview("VanillaRAG", vanilla_result)


=== VanillaRAG ===

QUESTION: Как Соня реагирует на признание Раскольникова в убийстве?

ANSWER:
Из представленных фрагментов можно сделать вывод, что Соня была глубоко потрясена и испытывала отчаяние, когда Раскольников намекал на своё преступление и позже признавался в нём. Её реакция была полна боли и сострадания.

CONTEXT: 10 chunk(s), first preview:
пока были вместе. Теперь же, только что разошлись, и та и другая стали об одном этом только и
думать. Соня припоминала, как вчера Свидригайлов сказал ей, что у Раскольникова две дороги —
Владимирка или… Она знала к тому же его тщеславие, заносчивость, самолюбие и неверие. «Неужели
же одно только малодушие и боязнь смерти могут заставить его жить?» — подумала она, наконец, в
отчаянии. Солнце между тем уже закатывалось. Она грустно стояла пред окном и пристально
смотрела в него,  — но в окно это была видна только одна капитальная небеленая стена соседнего
дома. Наконец, когда уж она дошла до совершенного убеждения в смерти несчастного, 

### Reranking RAG

In [14]:
from rag import RerankingRAG

rerank_rag = RerankingRAG(
    llm,
    retriever,
    TOP_K_DOCS,
    CANDIDATE_K,
    BOOK_TITLE,
    settings.reranker_model_name,
)
rerank_result = rerank_rag.run_rag_pipeline(question=TEST_QUESTION)
print_rag_preview("RerankingRAG", rerank_result)


=== RerankingRAG ===

QUESTION: Как Соня реагирует на признание Раскольникова в убийстве?

ANSWER:
Когда Раскольников начинает намекать на своё участие в убийстве, Соня испытывает сильный испуг и растерянность. Она не может сразу поверить в услышанное, и по её лицу пробегают судороги, она горько рыдает, закрыв руками лицо.

CONTEXT: 10 chunk(s), first preview:
—  Нет! нет! Не может быть, нет!  — как отчаянная, громко вскрикнула Соня, как будто ее вдруг
ножом ранили.  — Бог, бог такого ужаса не допустит!..
—  Других допускает же.
—  Нет, нет! Ее бог защитит, бог!..  — повторяла она, не помня себя.
—  Да, может, и бога-то совсем нет,  — с каким-то даже злорадством ответил Раскольников, засмеялся
и посмотрел на нее.
Лицо Сони вдруг страшно изменилось: по нем пробежали судороги. С невыразимым укором взглянула
она на него, хотела было что-то сказать, но ничего не могла выговорить и только вдруг горько-горько
зарыдала, закрыв руками лицо.
—  Вы говорите, У Катерины Ивановны ум мешается; у в

### HyDE + reranking

In [15]:
from rag import HydeRerankingRAG

hyde_rerank_rag = HydeRerankingRAG(
    llm,
    retriever,
    TOP_K_DOCS,
    CANDIDATE_K,
    BOOK_TITLE,
    settings.reranker_model_name,
)
hyde_rerank_result = hyde_rerank_rag.run_rag_pipeline(question=TEST_QUESTION)
print_rag_preview("HydeRerankingRAG", hyde_rerank_result)


=== HydeRerankingRAG ===

QUESTION: Как Соня реагирует на признание Раскольникова в убийстве?

ANSWER:
Когда Раскольников намекает Соне на своё участие в убийстве, она испытывает шок и страх. Соня не сразу понимает, что происходит, и реагирует с недоумением и ужасом: «Нет, нет, нет! — чуть слышно прошептала Соня». Она не может поверить в то, что слышит, и продолжает испытывать глубокие переживания по поводу случившегося.

CONTEXT: 10 chunk(s), first preview:
—  Нет! нет! Не может быть, нет!  — как отчаянная, громко вскрикнула Соня, как будто ее вдруг
ножом ранили.  — Бог, бог такого ужаса не допустит!..
—  Других допускает же.
—  Нет, нет! Ее бог защитит, бог!..  — повторяла она, не помня себя.
—  Да, может, и бога-то совсем нет,  — с каким-то даже злорадством ответил Раскольников, засмеялся
и посмотрел на нее.
Лицо Сони вдруг страшно изменилось: по нем пробежали судороги. С невыразимым укором взглянула
она на него, хотела было что-то сказать, но ничего не могла выговорить и только вд

### Query extend + reranking

In [16]:
from rag import RagQueryExtendRAG

query_extend_rag = RagQueryExtendRAG(
    llm,
    retriever,
    TOP_K_DOCS,
    CANDIDATE_K,
    BOOK_TITLE,
    settings.reranker_model_name,
)
query_extend_result = query_extend_rag.run_rag_pipeline(question=TEST_QUESTION)
print_rag_preview("RagQueryExtendRAG", query_extend_result)


=== RagQueryExtendRAG ===

QUESTION: Как Соня реагирует на признание Раскольникова в убийстве?

ANSWER:
Когда Раскольников начинает намекать на своё участие в убийстве, Соня испытывает шок и страх. Она не может сразу поверить в услышанное, и её реакция описывается как крайнее удивление и испуг: «Нет, не нашли», «Так как же вы про это знаете?», «Н-нет, — чуть слышно прошептала Соня».

CONTEXT: 10 chunk(s), first preview:
—  Нет! нет! Не может быть, нет!  — как отчаянная, громко вскрикнула Соня, как будто ее вдруг
ножом ранили.  — Бог, бог такого ужаса не допустит!..
—  Других допускает же.
—  Нет, нет! Ее бог защитит, бог!..  — повторяла она, не помня себя.
—  Да, может, и бога-то совсем нет,  — с каким-то даже злорадством ответил Раскольников, засмеялся
и посмотрел на нее.
Лицо Сони вдруг страшно изменилось: по нем пробежали судороги. С невыразимым укором взглянула
она на него, хотела было что-то сказать, но ничего не могла выговорить и только вдруг горько-горько
зарыдала, закрыв рука

## Метрики по CSV

Сводка по `evaluation_results/`. Для каждой пары (книга, `rag_type`) — один самый свежий CSV по дате в имени.

Сводная таблица: success rate в % (среднее трёх метрик DeepEval, порог 0.5). Нижняя строка — **macro average по всем 169 вопросам** (pooled), как в `report/report.tex`.

In [19]:
import re

import pandas as pd

RESULTS_DIR = EVAL_DIR / "evaluation_results"
SUCCESS_COLS = [
    "answer_relevancy_success",
    "contextual_recall_success",
    "faithfulness_success",
]


def _csv_timestamp(path: Path) -> str:
    m = re.search(r"_(\d{8}_\d{6})\.csv$", path.name)
    return m.group(1) if m else "0000"


def _bool_col(s: pd.Series) -> pd.Series:
    return s.astype(str).str.lower().isin(("true", "1", "yes"))


chosen: dict[tuple[str, str], Path] = {}
for path in sorted(RESULTS_DIR.glob("*.csv")):
    peek = pd.read_csv(path, nrows=1, usecols=["book_title", "rag_type"])
    key = (str(peek["book_title"].iloc[0]), str(peek["rag_type"].iloc[0]))
    ts = _csv_timestamp(path)
    if key not in chosen or ts > _csv_timestamp(chosen[key]):
        chosen[key] = path

frames: list[pd.DataFrame] = []
for path in chosen.values():
    df = pd.read_csv(path)
    df = df.drop_duplicates(subset=["question"], keep="first")
    for c in SUCCESS_COLS:
        df[c] = _bool_col(df[c])
    df["result_csv"] = path.name
    frames.append(df)

all_results = pd.concat(frames, ignore_index=True)

agg_rows = []
for keys, g in all_results.groupby(["book_title", "rag_type"], sort=True):
    n = len(g)
    row = {"book_title": keys[0], "rag_type": keys[1], "n_questions": n}
    for c in SUCCESS_COLS:
        row[c] = g[c].sum() / n if n else float("nan")
    row["mean_3_metrics"] = sum(row[c] for c in SUCCESS_COLS) / len(SUCCESS_COLS)
    row["csv"] = g["result_csv"].iloc[0]
    agg_rows.append(row)

summary = pd.DataFrame(agg_rows).sort_values(["book_title", "rag_type"]).reset_index(drop=True)

display_cols = [
    "book_title",
    "rag_type",
    "n_questions",
    "answer_relevancy_success",
    "contextual_recall_success",
    "faithfulness_success",
    "mean_3_metrics",
    "csv",
]
summary_fmt = summary[display_cols].copy()
for c in SUCCESS_COLS + ["mean_3_metrics"]:
    summary_fmt[c] = summary_fmt[c].map(lambda x: f"{x:.4f}")

summary_fmt

,book_title,rag_type,n_questions,answer_relevancy_success,contextual_recall_success,faithfulness_success,mean_3_metrics,csv
0,Герой нашего времени,hyde-reranking,45,0.8667,1.0000,1.0000,0.9556,Герой_нашего_времени_hyde-reranking_20260504_1...
1,Герой нашего времени,rag-query-extend,45,0.8444,1.0000,1.0000,0.9481,Герой_нашего_времени_rag-query-extend_20260504...
2,Герой нашего времени,reranking,45,0.8444,0.9778,1.0000,0.9407,Герой_нашего_времени_reranking_20260504_154918...
3,Герой нашего времени,vanilla,45,0.8667,0.9333,1.0000,0.9333,Герой_нашего_времени_vanilla_20260517_165031.csv
4,Капитанская дочка,hyde-reranking,37,0.9459,0.8378,0.9730,0.9189,Капитанская_дочка_hyde-reranking_20260504_1129...
5,Капитанская дочка,rag-query-extend,37,0.9730,0.8649,1.0000,0.9459,Капитанская_дочка_rag-query-extend_20260504_12...
6,Капитанская дочка,reranking,37,0.9730,0.8108,0.9459,0.9099,Капитанская_дочка_reranking_20260504_105916.csv
7,Капитанская дочка,vanilla,37,0.9730,0.8108,0.9189,0.9009,Капитанская_дочка_vanilla_20260517_172010.csv
8,Отцы и дети,hyde-reranking,43,0.9302,1.0000,1.0000,0.9767,Отцы_и_дети_hyde-reranking_20260504_130753.csv
9,Отцы и дети,rag-query-extend,43,0.9302,0.9767,1.0000,0.9690,Отцы_и_дети_rag-query-extend_20260504_133824.csv


In [20]:
RAG_COL_ORDER = ["vanilla", "reranking", "hyde-reranking", "rag-query-extend"]

pivot_mean = summary.pivot_table(
    index="book_title",
    columns="rag_type",
    values="mean_3_metrics",
    aggfunc="first",
)[RAG_COL_ORDER]

pooled = []
for rag_type in RAG_COL_ORDER:
    g = all_results.loc[all_results["rag_type"] == rag_type]
    n = len(g)
    m3 = sum(g[c].sum() / n for c in SUCCESS_COLS) / len(SUCCESS_COLS)
    pooled.append(m3)
pivot_mean.loc["Macro average (все книги)"] = pooled

(pivot_mean * 100).round(1)

rag_type,vanilla,reranking,hyde-reranking,rag-query-extend
book_title,,,,
Герой нашего времени,93.3,94.1,95.6,94.8
Капитанская дочка,90.1,91.0,91.9,94.6
Отцы и дети,89.1,95.3,97.7,96.9
Преступление и наказание,92.4,93.2,93.2,93.2
Macro average (все книги),91.3,93.5,94.7,94.9
